# 근접전계 → 원전계 변환 프로그램

이 노트북은 프로브를 이용한 근접전계 스캔 데이터를 기반으로 원전계 패턴을 계산하는 전자기장 변환 프로그램입니다.

## 주요 기능

1. **CSV 데이터 로드**: 2차원 평면 좌표별 크기/위상 데이터 로드
2. **원전계 변환**: Plane Wave Spectrum 방법 사용
3. **Back Projection**: 원전계 → 근접전계 역변환 및 검증
4. **시각화**: 원본/변환/비교 데이터 시각화

## 사용 방법

각 셀을 순서대로 실행하세요. 파라미터는 필요에 따라 조정할 수 있습니다.

## 1. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

print("라이브러리 임포트 완료!")

## 2. 파라미터 설정

여기서 변환에 필요한 파라미터를 설정합니다. 필요에 따라 값을 수정하세요.

In [ ]:
# ==================== 파라미터 설정 ====================

# 주파수 (Hz)
frequency = 10e9  # 10 GHz

# 근접전계 측정 평면의 Z 좌표 (m)
z_distance = 0.01  # 10 mm

# 원전계 계산 거리 (m)
far_field_distance = 1.0  # 1 m

# Theta 각도 범위 (시작, 끝, 스텝) 단위: 도
theta_start, theta_end, theta_step = 0, 90, 1

# Phi 각도 범위 (시작, 끝, 스텝) 단위: 도
phi_start, phi_end, phi_step = 0, 360, 5

# 계산된 값
c = 3e8  # 빛의 속도
wavelength = c / frequency
k0 = 2 * np.pi / wavelength

print("=" * 60)
print(f"주파수: {frequency/1e9:.2f} GHz")
print(f"파장: {wavelength*1000:.2f} mm")
print(f"근접전계 스캔 거리: {z_distance*1000:.1f} mm")
print(f"원전계 계산 거리: {far_field_distance:.2f} m")
print(f"Theta 범위: {theta_start}° ~ {theta_end}° (간격: {theta_step}°)")
print(f"Phi 범위: {phi_start}° ~ {phi_end}° (간격: {phi_step}°)")
print("=" * 60)

## 3. 예제 데이터 생성 (선택사항)

실제 측정 데이터가 없는 경우, 시뮬레이션된 예제 데이터를 생성할 수 있습니다.
실제 CSV 파일이 있다면 이 셀은 건너뛰고 다음 셀에서 직접 로드하세요.

In [ ]:
def generate_dipole_field(x, y, z, freq, dipole_moment=1.0):
    """전기 쌍극자의 근접전계 계산"""
    c = 3e8
    wavelength = c / freq
    k = 2 * np.pi / wavelength
    mu0 = 4 * np.pi * 1e-7
    epsilon0 = 8.854e-12
    
    r = np.sqrt(x**2 + y**2 + z**2)
    dz = z
    
    Ez = (dipole_moment / (4 * np.pi * epsilon0)) * (
        (3 * dz * dz / r**5 - 1 / r**3) +
        1j * k * (3 * dz * dz / r**4 - 1 / r**2) -
        k**2 * dz * dz / r**3
    ) * np.exp(-1j * k * r)
    
    return Ez

# 예제 데이터 생성
x_range = np.arange(-0.08, 0.08, 0.004)
y_range = np.arange(-0.08, 0.08, 0.004)
X, Y = np.meshgrid(x_range, y_range)
Z = np.ones_like(X) * z_distance

# 전기장 계산
field = generate_dipole_field(X, Y, Z, frequency)

# 노이즈 추가
snr_db = 50
signal_power = np.mean(np.abs(field)**2)
noise_power = signal_power / (10**(snr_db/10))
noise = (np.random.normal(0, np.sqrt(noise_power/2), field.shape) + 
         1j * np.random.normal(0, np.sqrt(noise_power/2), field.shape))
field = field + noise

# CSV 파일로 저장
magnitude = np.abs(field)
phase_deg = np.rad2deg(np.angle(field))

data = {
    'x': X.flatten(),
    'y': Y.flatten(),
    'magnitude': magnitude.flatten(),
    'phase': phase_deg.flatten()
}

df = pd.DataFrame(data)
df.to_csv('near_field_data.csv', index=False)

print(f"예제 데이터 생성 완료!")
print(f"그리드 크기: {len(x_range)} x {len(y_range)}")
print(f"총 데이터 포인트: {len(df)}")
print(f"파일 저장: near_field_data.csv")

## 4. CSV 데이터 로드

CSV 파일에서 근접전계 측정 데이터를 로드합니다.
CSV 형식: `x`, `y`, `magnitude`, `phase` 컬럼 필요

In [ ]:
# CSV 파일 로드
csv_file = 'near_field_data.csv'
df = pd.read_csv(csv_file)

print(f"CSV 파일 로드: {csv_file}")
print(f"\n데이터 미리보기:")
print(df.head())

# 데이터 재구성
x_coords = np.unique(df['x'].values)
y_coords = np.unique(df['y'].values)

nx = len(x_coords)
ny = len(y_coords)

magnitude = df['magnitude'].values.reshape(ny, nx)
phase_deg = df['phase'].values.reshape(ny, nx)
phase_rad = np.deg2rad(phase_deg)

# 복소수 전기장
near_field_data = magnitude * np.exp(1j * phase_rad)
X_grid, Y_grid = np.meshgrid(x_coords, y_coords)

print(f"\n그리드 크기: {nx} x {ny} 포인트")
print(f"X 범위: {x_coords[0]:.4f} ~ {x_coords[-1]:.4f} m")
print(f"Y 범위: {y_coords[0]:.4f} ~ {y_coords[-1]:.4f} m")

## 5. 원본 근접전계 데이터 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 크기 플롯
im1 = axes[0].contourf(X_grid * 1000, Y_grid * 1000, magnitude, 
                       levels=50, cmap='hot')
axes[0].set_xlabel('X (mm)')
axes[0].set_ylabel('Y (mm)')
axes[0].set_title('Near-Field Magnitude')
axes[0].set_aspect('equal')
plt.colorbar(im1, ax=axes[0], label='Magnitude (V/m)')

# 위상 플롯
im2 = axes[1].contourf(X_grid * 1000, Y_grid * 1000, phase_deg, 
                       levels=50, cmap='hsv')
axes[1].set_xlabel('X (mm)')
axes[1].set_ylabel('Y (mm)')
axes[1].set_title('Near-Field Phase')
axes[1].set_aspect('equal')
plt.colorbar(im2, ax=axes[1], label='Phase (deg)')

plt.tight_layout()
plt.show()

print("원본 근접전계 데이터 시각화 완료")

## 6. 원전계 변환 (Plane Wave Spectrum 방법)

In [ ]:
print("원전계 변환 수행 중...")

dx = x_coords[1] - x_coords[0]
dy = y_coords[1] - y_coords[0]

# 2D FFT를 이용한 스펙트럼 계산
spectrum = np.fft.fft2(near_field_data)
spectrum = np.fft.fftshift(spectrum)

# 공간 주파수
kx = np.fft.fftfreq(nx, dx) * 2 * np.pi
ky = np.fft.fftfreq(ny, dy) * 2 * np.pi
kx = np.fft.fftshift(kx)
ky = np.fft.fftshift(ky)
KX, KY = np.meshgrid(kx, ky)

# 전파 상수
kz_squared = k0**2 - KX**2 - KY**2
KZ = np.sqrt(kz_squared.astype(complex))

# Evanescent wave 필터링
propagating_mask = (kz_squared > 0)
KZ[~propagating_mask] = 0

# 원전계 거리로 전파
propagation_distance = far_field_distance - z_distance
propagation_factor = np.exp(1j * KZ * propagation_distance)
far_spectrum = spectrum * propagation_factor

# 원전계 각도별 패턴 계산
theta = np.deg2rad(np.arange(theta_start, theta_end, theta_step))
phi = np.deg2rad(np.arange(phi_start, phi_end, phi_step))

THETA, PHI = np.meshgrid(theta, phi)
far_field_pattern = np.zeros_like(THETA, dtype=complex)

# 각 방향에 대해 스펙트럼 샘플링
for i, th in enumerate(theta):
    for j, ph in enumerate(phi):
        kx_sample = k0 * np.sin(th) * np.cos(ph)
        ky_sample = k0 * np.sin(th) * np.sin(ph)
        
        idx_x = np.argmin(np.abs(kx - kx_sample))
        idx_y = np.argmin(np.abs(ky - ky_sample))
        
        if propagating_mask[idx_y, idx_x]:
            far_field_pattern[j, i] = far_spectrum[idx_y, idx_x]

print("원전계 변환 완료!")
print(f"원전계 패턴 크기: {THETA.shape}")

## 7. 원전계 패턴 시각화

In [ ]:
# 원전계 크기를 dB로 변환
magnitude_db = 20 * np.log10(np.abs(far_field_pattern) + 1e-10)
magnitude_db -= np.max(magnitude_db)  # 정규화

fig = plt.figure(figsize=(16, 5))

# 1. 2D 히트맵
ax1 = fig.add_subplot(131)
im = ax1.contourf(np.rad2deg(THETA), np.rad2deg(PHI), magnitude_db, 
                  levels=50, cmap='jet')
ax1.set_xlabel('Theta (deg)')
ax1.set_ylabel('Phi (deg)')
ax1.set_title('Far-Field Pattern (dB)')
plt.colorbar(im, ax=ax1, label='Relative Magnitude (dB)')

# 2. Phi=0 평면 극좌표 플롯
ax2 = fig.add_subplot(132, projection='polar')
phi_0_idx = len(PHI) // 2
theta_cut = THETA[phi_0_idx, :]
pattern_cut = magnitude_db[phi_0_idx, :]
ax2.plot(theta_cut, pattern_cut)
ax2.set_theta_zero_location('N')
ax2.set_theta_direction(-1)
ax2.set_title('Far-Field Pattern (Phi=0°)')
ax2.set_ylim([-40, 0])
ax2.grid(True)

# 3. 3D 표면 플롯
ax3 = fig.add_subplot(133, projection='3d')
X_3d = np.abs(far_field_pattern) * np.sin(THETA) * np.cos(PHI)
Y_3d = np.abs(far_field_pattern) * np.sin(THETA) * np.sin(PHI)
Z_3d = np.abs(far_field_pattern) * np.cos(THETA)

surf = ax3.plot_surface(X_3d, Y_3d, Z_3d, cmap='jet', alpha=0.8)
ax3.set_xlabel('X')
ax3.set_ylabel('Y')
ax3.set_zlabel('Z')
ax3.set_title('Far-Field 3D Pattern')

plt.tight_layout()
plt.show()

print("원전계 패턴 시각화 완료")

## 8. Back Projection (원전계 → 근접전계 역변환)

In [ ]:
print("Back Projection 수행 중...")

# 스펙트럼에서 역전파
back_propagation_factor = np.exp(-1j * KZ * propagation_distance)
back_spectrum = far_spectrum * back_propagation_factor

# 역 FFT
back_spectrum = np.fft.ifftshift(back_spectrum)
back_projected = np.fft.ifft2(back_spectrum)

print("Back Projection 완료!")

## 9. 원본 vs Back Projection 비교

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

original_mag = np.abs(near_field_data)
original_phase = np.angle(near_field_data, deg=True)

back_mag = np.abs(back_projected)
back_phase = np.angle(back_projected, deg=True)

error_mag = np.abs(original_mag - back_mag)
error_phase = np.abs(original_phase - back_phase)

# 원본 크기
im1 = axes[0, 0].contourf(X_grid * 1000, Y_grid * 1000, original_mag, 
                          levels=50, cmap='hot')
axes[0, 0].set_title('Original Near-Field - Magnitude')
axes[0, 0].set_xlabel('X (mm)')
axes[0, 0].set_ylabel('Y (mm)')
axes[0, 0].set_aspect('equal')
plt.colorbar(im1, ax=axes[0, 0])

# 원본 위상
im2 = axes[0, 1].contourf(X_grid * 1000, Y_grid * 1000, original_phase, 
                          levels=50, cmap='hsv')
axes[0, 1].set_title('Original Near-Field - Phase')
axes[0, 1].set_xlabel('X (mm)')
axes[0, 1].set_ylabel('Y (mm)')
axes[0, 1].set_aspect('equal')
plt.colorbar(im2, ax=axes[0, 1])

# 크기 오차
im3 = axes[0, 2].contourf(X_grid * 1000, Y_grid * 1000, error_mag, 
                          levels=50, cmap='viridis')
axes[0, 2].set_title('Magnitude Error')
axes[0, 2].set_xlabel('X (mm)')
axes[0, 2].set_ylabel('Y (mm)')
axes[0, 2].set_aspect('equal')
plt.colorbar(im3, ax=axes[0, 2])

# Back projection 크기
im4 = axes[1, 0].contourf(X_grid * 1000, Y_grid * 1000, back_mag, 
                          levels=50, cmap='hot')
axes[1, 0].set_title('Back Projection - Magnitude')
axes[1, 0].set_xlabel('X (mm)')
axes[1, 0].set_ylabel('Y (mm)')
axes[1, 0].set_aspect('equal')
plt.colorbar(im4, ax=axes[1, 0])

# Back projection 위상
im5 = axes[1, 1].contourf(X_grid * 1000, Y_grid * 1000, back_phase, 
                          levels=50, cmap='hsv')
axes[1, 1].set_title('Back Projection - Phase')
axes[1, 1].set_xlabel('X (mm)')
axes[1, 1].set_ylabel('Y (mm)')
axes[1, 1].set_aspect('equal')
plt.colorbar(im5, ax=axes[1, 1])

# 위상 오차
im6 = axes[1, 2].contourf(X_grid * 1000, Y_grid * 1000, error_phase, 
                          levels=50, cmap='viridis')
axes[1, 2].set_title('Phase Error')
axes[1, 2].set_xlabel('X (mm)')
axes[1, 2].set_ylabel('Y (mm)')
axes[1, 2].set_aspect('equal')
plt.colorbar(im6, ax=axes[1, 2])

plt.tight_layout()
plt.show()

print("비교 시각화 완료")

## 10. 오차 분석

In [ ]:
print("=" * 60)
print("Back Projection 오차 분석")
print("=" * 60)
print(f"크기 평균 오차: {np.mean(error_mag):.6f}")
print(f"크기 최대 오차: {np.max(error_mag):.6f}")
print(f"크기 상대 오차: {np.mean(error_mag/original_mag)*100:.2f}%")
print(f"위상 평균 오차: {np.mean(error_phase):.2f}°")
print(f"위상 최대 오차: {np.max(error_phase):.2f}°")
print("=" * 60)

## 요약

이 노트북에서는 다음 작업을 수행했습니다:

1. ✅ 근접전계 데이터 로드 (CSV)
2. ✅ 원전계 변환 (Plane Wave Spectrum 방법)
3. ✅ 원전계 패턴 시각화 (2D, 극좌표, 3D)
4. ✅ Back Projection 수행
5. ✅ 원본 vs Back Projection 비교 및 오차 분석

### 파라미터 조정 가이드

결과를 개선하거나 다른 설정을 시도하려면:
- **주파수**: 다른 주파수 대역 테스트
- **각도 해상도**: `theta_step`, `phi_step` 값 조정
- **원전계 거리**: `far_field_distance` 변경
- **데이터 소스**: 실제 측정 CSV 파일 사용

### 다음 단계

- 다양한 안테나 구조 테스트
- 더 높은 해상도 스캔
- 다중 주파수 분석
- 실제 측정 데이터와 비교